In [ ]:
import torch
import numpy as np
from torch.utils.tensorboard import SummaryWriter
import datetime
from collections import Counter

current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
log_dir = f"runs/asl_data_eda_{current_time}"
writer = SummaryWriter(log_dir)
print(f"Logging to {log_dir}")


In [2]:
from dataset import ASLFeatureDataset
from torch.utils.data import DataLoader

train_dataset = ASLFeatureDataset(root_dir="../wlasl100_features/train", max_frames=100)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True, num_workers=4, persistent_workers=True
)

validation_dataset = ASLFeatureDataset(
    root_dir="../wlasl100_features/val", max_frames=100
)

validation_loader = DataLoader(
    validation_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4,
    persistent_workers=True,
)

features, labels, lengths = next(iter(train_loader))
assert (lengths <= features.shape[1]).all(), (
    "Some lengths are greater than the sequence length"
)
print(lengths.max(), features.shape[1])

tensor(100) 100


### Tensorboard logging


In [3]:
import matplotlib.pyplot as plt

label_counts = Counter(train_dataset.labels)

classes = [train_dataset.classes[label] for label in sorted(label_counts)]
counts = [label_counts[label] for label in sorted(label_counts)]

fig, ax = plt.subplots(figsize=(24, 6))
ax.bar(classes, counts)
ax.set_xlabel("Class")
ax.set_ylabel("Count")
ax.set_title("Class Counts")
plt.xticks(rotation=90, fontsize=6)
fig.tight_layout()
writer.add_figure("Data_Distribution/Class_Counts", fig, global_step=0)

# Sequence lengths and feature values
TARGET_SAMPLES = 10_000
FEATURE_DIM = 126
sample_prob = TARGET_SAMPLES / (
    len(train_dataset) * train_dataset.max_frames * FEATURE_DIM
)

all_lengths = []
feature_chunks = []
x_chunks, y_chunks, z_chunks = [], [], []

for i in range(len(train_dataset)):
    padded_tensor, _, actual_frames = train_dataset[i]
    all_lengths.append(actual_frames)

    valid_frames = padded_tensor[:actual_frames, :]
    feature_chunks.append(valid_frames[torch.rand(valid_frames.shape) < sample_prob])

    x_coords = valid_frames[:, 0::3]
    y_coords = valid_frames[:, 1::3]
    z_coords = valid_frames[:, 2::3]
    coord_mask = torch.rand(x_coords.shape) < sample_prob
    x_chunks.append(x_coords[coord_mask])
    y_chunks.append(y_coords[coord_mask])
    z_chunks.append(z_coords[coord_mask])

lengths_tensor = torch.tensor(all_lengths)
features_tensor = torch.cat(feature_chunks)
x_tensor = torch.cat(x_chunks)
y_tensor = torch.cat(y_chunks)
z_tensor = torch.cat(z_chunks)
# if features_tensor.numel() > TARGET_SAMPLES:
#     features_tensor = features_tensor[
#         torch.randperm(features_tensor.numel())[:TARGET_SAMPLES]
#     ]

all_zero_count = ((x_tensor == 0) & (y_tensor == 0) & (z_tensor == 0)).sum().item()
print(len(x_tensor) + len(y_tensor) + len(z_tensor))

writer.add_histogram("Data_Distribution/Sequence_Lengths", lengths_tensor, 0)
writer.add_histogram("Data_Distribution/Feature_Values", features_tensor, 0)
writer.add_histogram("Data_Distribution/X_Values", x_tensor, 0)
writer.add_histogram("Data_Distribution/Y_Values", y_tensor, 0)
writer.add_histogram("Data_Distribution/Z_Values", z_tensor, 0)
writer.add_scalar("Data_Distribution/All_Axis_Zero_Count", all_zero_count, 0)

writer.flush()
writer.close()


6351


In [4]:
import torch.nn as nn
import torch.optim as optim
from model import ASLSequentialProcessor

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Using device: {device}")

model = ASLSequentialProcessor(
    input_size=126, hidden_size=256, num_layers=2, num_classes=100
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)

current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
writer = SummaryWriter(f"runs/asl_training_phase1_{current_time}")


Using device: mps


In [5]:
current_epoch = 0

In [6]:
epochs = 50

for epoch in range(epochs):
    model.train()
    running_train_loss = 0.0

    for batch_idx, (features, labels, lengths) in enumerate(train_loader):
        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features, lengths)

        loss = criterion(outputs, labels)

        loss.backward()

        # tensorboard log gradient norms
        # log every 100 batches
        if batch_idx % 10 == 9:
            total_norm = 0
            for p in model.parameters():
                if p.grad is not None:
                    param_norm = p.grad.data.norm(2)
                    total_norm += param_norm.item() ** 2
            total_norm = total_norm**0.5
            writer.add_scalar(
                "Gradients/Norm",
                total_norm,
                current_epoch * len(train_loader) + batch_idx,
            )

            print(
                f"Epoch {current_epoch + 1} | Batch {batch_idx + 1} | Train Loss: {loss.item():.4f}"
            )

        optimizer.step()

        running_train_loss += loss.item()

    avg_train_loss = running_train_loss / len(train_loader)

    # Validation
    model.eval()
    running_val_loss = 0.0
    correct_predictions = 0
    total_samples = 0

    with torch.no_grad():
        for features, labels, lengths in validation_loader:
            features = features.to(device)
            labels = labels.to(device)

            outputs = model(features, lengths)
            loss = criterion(outputs, labels)
            running_val_loss += loss.item()

            _, predicted = torch.max(outputs.data, 1)
            total_samples += labels.size(0)
            correct_predictions += (predicted == labels).sum().item()

    avg_val_loss = running_val_loss / len(validation_loader)
    val_accuracy = correct_predictions / total_samples * 100

    writer.add_scalars(
        "Loss/Phase1",
        {"Train": avg_train_loss, "Validation": avg_val_loss},
        current_epoch,
    )

    writer.add_scalar("Accuracy/Validation", val_accuracy, current_epoch)

    print(
        f"Epoch {current_epoch + 1} | Train Loss: {avg_train_loss:.4f} | Validation Loss: {avg_val_loss:.4f} | Validation Accuracy: {val_accuracy:.2f}%"
    )
    current_epoch += 1

writer.close()
print("Phase 1 training complete")

Epoch 1 | Batch 10 | Train Loss: 4.6242
Epoch 1 | Batch 20 | Train Loss: 4.6122
Epoch 1 | Train Loss: 4.6102 | Validation Loss: 4.5912 | Validation Accuracy: 1.21%
Epoch 2 | Batch 10 | Train Loss: 4.5184
Epoch 2 | Batch 20 | Train Loss: 4.5373
Epoch 2 | Train Loss: 4.5617 | Validation Loss: 4.6319 | Validation Accuracy: 1.82%
Epoch 3 | Batch 10 | Train Loss: 4.6551
Epoch 3 | Batch 20 | Train Loss: 4.6931
Epoch 3 | Train Loss: 4.6289 | Validation Loss: 4.6036 | Validation Accuracy: 0.61%
Epoch 4 | Batch 10 | Train Loss: 4.5736
Epoch 4 | Batch 20 | Train Loss: 4.6095
Epoch 4 | Train Loss: 4.5839 | Validation Loss: 4.5914 | Validation Accuracy: 1.21%
Epoch 5 | Batch 10 | Train Loss: 4.6095
Epoch 5 | Batch 20 | Train Loss: 4.3492
Epoch 5 | Train Loss: 4.4765 | Validation Loss: 4.4382 | Validation Accuracy: 0.61%
Epoch 6 | Batch 10 | Train Loss: 4.2072
Epoch 6 | Batch 20 | Train Loss: 4.7011
Epoch 6 | Train Loss: 4.4664 | Validation Loss: 4.7015 | Validation Accuracy: 0.61%
Epoch 7 | Batch 